# SolanaSentinel — Entrenamiento del Modelo v2

## Cambios vs v1
| | v1 | v2 |
|---|---|---|
| **Dataset** | ~22k tokens | 35,720 tokens |
| **Label** | ratio_1h >= 1.2 (precio a 1h exacta) | outcome_max_gain_pct >= 20% (pico real) |
| **Por que** | Si el token pica y baja antes de 1h: falso negativo | El sniper sale al PRIMER +20%, no a la 1h |
| **Features nuevas** | — | f_bc_progress, f_dow_sin, f_dow_cos |
| **PR-AUC** | 0.108 | 0.152 (+40%) |

## El label correcto para el sniper

El sniper usa TP=20%, SL=10% y monitorea continuamente.
Cuando el precio sube +20%, sale inmediatamente — **no espera 1h**.

`outcome_max_gain_pct` = maximo ganado en cualquier momento de vida del token.
Si `outcome_max_gain_pct >= 20`, el sniper habria salido en TP en algun momento.

### Metricas correctas para imbalance (4.6% positivos):
| Metrica | Por que sirve |
|---|---|
| **PR-AUC** | Mide precision+recall solo en la clase positiva |
| **ROC-AUC** | Mide capacidad de ranking |
| **Win rate por score bucket** | Cuando score>=75, cuantos llegan al TP? |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    classification_report, confusion_matrix
)
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost disponible')
except ImportError:
    HAS_XGB = False
    print('XGBoost no instalado — pip install xgboost')

plt.style.use('dark_background')
ACCENT  = '#00d4ff'
GREEN   = '#00ff88'
RED     = '#ff4444'
PURPLE  = '#a78bfa'
YELLOW  = '#fbbf24'

DB_PATH    = Path('../backend/data/sentinel.db')
MODELS_DIR = Path('../backend/data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
Path('charts').mkdir(exist_ok=True)

LABEL_THRESHOLD = 20.0        # outcome_max_gain_pct >= 20 (TP-aligned)
LABEL_NAME      = 'pump_tp_20pct'
LABEL_COL       = 'outcome_max_gain_pct'
BC_GRADUATION   = 85.0         # SOL para graduarse en pump.fun

assert DB_PATH.exists(), f'DB no encontrada: {DB_PATH}'
print(f'Label: precio_1h / precio_detección >= {LABEL_THRESHOLD}')

## 1. Carga y features

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql_query("""
    SELECT id, source, detected_at,
           initial_liquidity, market_cap, volume_1h,
           COALESCE(NULLIF(price_usd, 0), latest_price_usd) AS price_usd,
           bonding_curve_complete, bonding_curve_real_sol,
           bonding_curve_mc_sol, bonding_curve_price_sol,
           risk_score, risk_level,
           outcome_max_gain_pct, outcome_price_1h
    FROM detected_tokens
    WHERE outcome_complete = 1
      AND outcome_max_gain_pct IS NOT NULL
""", conn)
conn.close()

eps = 1e-12
df  = df_raw.copy()

# Label v2: el token alguna vez subio >=20% desde deteccion?
df['label']    = (df['outcome_max_gain_pct'] >= LABEL_THRESHOLD).astype(int)

# Features existentes
df['f_log_mc']        = np.log1p(df['market_cap'].fillna(0).clip(lower=0))
df['f_log_liq']       = np.log1p(df['initial_liquidity'].fillna(0).clip(lower=0))
df['f_log_price']     = np.log1p(df['price_usd'].fillna(0).clip(lower=0))
df['f_liq_mc_ratio']  = df['initial_liquidity'].fillna(0) / (df['market_cap'].fillna(0) + eps)
df['f_on_curve']      = (df['bonding_curve_complete'].fillna(1) == 0).astype(float)
df['f_log_bc_sol']    = np.log1p(df['bonding_curve_real_sol'].fillna(0).clip(lower=0))
df['f_log_bc_mc_sol'] = np.log1p(df['bonding_curve_mc_sol'].fillna(0).clip(lower=0))
df['f_risk_score']    = df['risk_score'].fillna(50)
df['f_is_safe']       = (df['risk_level'].isin(['safe', 'low'])).astype(float)
df['f_is_pumpfun']    = (df['source'] == 'pumpfun').astype(float)
df['detected_dt']     = pd.to_datetime(df['detected_at'])
df['f_hour_sin']      = np.sin(2 * np.pi * df['detected_dt'].dt.hour / 24)
df['f_hour_cos']      = np.cos(2 * np.pi * df['detected_dt'].dt.hour / 24)

# NUEVAS features v2
df['f_bc_progress']   = (df['bonding_curve_real_sol'].fillna(0) / BC_GRADUATION).clip(0, 1)
df['f_dow_sin']       = np.sin(2 * np.pi * df['detected_dt'].dt.dayofweek / 7)
df['f_dow_cos']       = np.cos(2 * np.pi * df['detected_dt'].dt.dayofweek / 7)

FEATURE_COLS = [
    'f_log_mc', 'f_log_liq', 'f_log_price', 'f_liq_mc_ratio',
    'f_on_curve', 'f_log_bc_sol', 'f_log_bc_mc_sol',
    'f_risk_score', 'f_is_safe', 'f_is_pumpfun',
    'f_hour_sin', 'f_hour_cos',
    'f_bc_progress',        # NUEVA v2
    'f_dow_sin', 'f_dow_cos',   # NUEVA v2
]

n_pos  = df['label'].sum()
n_neg  = (df['label'] == 0).sum()
base   = n_pos / len(df)
print(f'Total:      {len(df):,}')
print(f'Positivos:  {n_pos} ({base*100:.2f}%)  <- tokens que hubieran activado TP')
print(f'Negativos:  {n_neg} ({n_neg/len(df)*100:.2f}%)')
print(f'Imbalance:  {n_neg/n_pos:.1f}:1')
print(f'Features:   {len(FEATURE_COLS)} (+3 vs v1)')
print(f'
Comparacion con v1:')
print(f'  v1: ~22k samples, label=ratio_1h>=1.2')
print(f'  v2: {len(df):,} samples, label=max_gain>=20%')

## 2. Split temporal (80% train / 20% test)

In [ ]:
# Split por tiempo: el modelo solo ve el pasado, se evalúa en el futuro
df_s = df.sort_values('detected_at').reset_index(drop=True)
X    = df_s[FEATURE_COLS].fillna(0).values
y    = df_s['label'].values

split = int(len(df_s) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train: {len(X_train):,}  →  {y_train.sum()} positivos ({y_train.mean()*100:.1f}%)')
print(f'Test:  {len(X_test):,}   →  {y_test.sum()} positivos ({y_test.mean()*100:.1f}%)')

## 3. Entrenamiento

`class_weight='balanced'` hace que el modelo penalice 8.4x más los errores sobre tokens que sí pumpearon.
Esto es equivalente a SMOTE pero sin generar datos sintéticos que pueden introducir ruido.

In [ ]:
models = {}

# Random Forest con class_weight balanced
print('Entrenando Random Forest...')
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    class_weight='balanced_subsample',  # balanced por árbol, más robusto que balanced global
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
models['RandomForest'] = rf
print('  done')

if HAS_XGB:
    print('Entrenando XGBoost...')
    scale_pos_w = (y_train == 0).sum() / (y_train == 1).sum()
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_w,   # penaliza 8.4x los falsos negativos
        eval_metric='aucpr',            # optimiza PR-AUC directamente
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    print('  done')

print(f'\nModelos entrenados: {list(models.keys())}')

## 4. Evaluación — métricas correctas para imbalance

**No miramos accuracy.** Las métricas relevantes son:
- **ROC-AUC**: ¿puede el modelo ordenar correctamente (pump arriba, no-pump abajo)?
- **PR-AUC**: ¿cuánta precision mantiene a distintos niveles de recall?
- **Base rate**: 10.7% → un modelo aleatorio tiene PR-AUC = 0.107

In [ ]:
results = {}
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    ap    = average_precision_score(y_test, proba)
    results[name] = {'proba': proba, 'auc': auc, 'ap': ap}
    print(f'{name:<20}  ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}  '
          f'(base={y_test.mean():.3f}  lift={ap/y_test.mean():.1f}x)')

print(f'\nBase rate (modelo aleatorio): ROC-AUC=0.5000  PR-AUC={y_test.mean():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = [ACCENT, GREEN, PURPLE]

# ROC
ax = axes[0]
ax.plot([0,1],[0,1], 'w--', lw=0.8, label='Random (AUC=0.50)')
for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{name} (AUC={r["auc"]:.3f})')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.grid(alpha=0.2)

# Precision-Recall
ax = axes[1]
base_rate = y_test.mean()
ax.axhline(base_rate, color='white', ls='--', lw=0.8,
           label=f'Random (PR-AUC≈{base_rate:.3f})')
for (name, r), c in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, r['proba'])
    ax.plot(rec, prec, color=c, lw=2, label=f'{name} (PR-AUC={r["ap"]:.3f})')
ax.fill_between([0,1], [base_rate, base_rate], alpha=0.05, color='white')
ax.set_title('Precision-Recall Curve\n(área sobre la línea blanca = mejora real sobre random)')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('charts/11_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Win rate por score bucket — la visualización clave

**La pregunta que importa para el sniper:**
> *"De todos los tokens que el modelo puntúa entre 80 y 90, ¿cuántos realmente subieron +20%?"*

Si sin modelo el 10.7% pumpe, y con score >80 el 40% pumpe → el modelo es útil.

Usamos el mejor modelo (mayor PR-AUC).

In [ ]:
best_name  = max(results, key=lambda k: results[k]['ap'])
best_model = models[best_name]
best_proba = results[best_name]['proba']
print(f'Mejor modelo: {best_name}  PR-AUC={results[best_name]["ap"]:.4f}')

# ── Win rate por decil de score ───────────────────────────────────────────────
df_test_eval = pd.DataFrame({
    'score':   best_proba,
    'pumped':  y_test,
    'ratio':   df_s.iloc[split:]['ratio_1h'].values
})

# Deciles del score
df_test_eval['decile'] = pd.qcut(df_test_eval['score'], q=10,
                                  labels=False, duplicates='drop')
decile_stats = df_test_eval.groupby('decile').agg(
    n         = ('pumped', 'count'),
    win_rate  = ('pumped', 'mean'),
    score_min = ('score', 'min'),
    score_max = ('score', 'max'),
    ratio_med = ('ratio', 'median')
).reset_index()
decile_stats['score_range'] = decile_stats.apply(
    lambda r: f'{r["score_min"]*100:.0f}-{r["score_max"]*100:.0f}', axis=1
)

print('\nWin rate (+20% en 1h) por decil de score ML:')
print(f'{"Decil":<8} {"Score range":<14} {"N tokens":<10} {"Win rate":<12} {"vs base"}')
print('-' * 55)
for _, row in decile_stats.iterrows():
    lift = row['win_rate'] / base_rate if base_rate > 0 else 0
    bar  = '█' * int(row['win_rate'] * 40)
    flag = ' ✅' if row['win_rate'] >= base_rate * 2 else ''
    print(f'{int(row["decile"])+1:<8} {row["score_range"]:<14} {int(row["n"]):<10} '
          f'{row["win_rate"]*100:5.1f}%  {lift:.1f}x  {bar}{flag}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Win rate por decil
ax = axes[0]
bar_colors = [GREEN if w >= base_rate*3 else ACCENT if w >= base_rate else RED
              for w in decile_stats['win_rate']]
bars = ax.bar(range(len(decile_stats)), decile_stats['win_rate'] * 100,
              color=bar_colors, edgecolor='none', alpha=0.85)
ax.axhline(base_rate * 100, color='white', ls='--', lw=1.5,
           label=f'Base rate ({base_rate*100:.1f}%)')
ax.axhline(base_rate * 300, color=GREEN, ls=':', lw=1,
           label=f'3x base ({base_rate*300:.1f}%)')
ax.set_xticks(range(len(decile_stats)))
ax.set_xticklabels(decile_stats['score_range'], rotation=45, ha='right', fontsize=8)
ax.set_title(f'Win rate por decil de score — {best_name}\n'
             f'(verde = 3x base rate ≥{base_rate*200:.0f}%)')
ax.set_xlabel('Score ML (rango)')
ax.set_ylabel('% tokens que suben +20% en 1h')
ax.legend()
for bar, val in zip(bars, decile_stats['win_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=8)

# Nº de tokens por decil
ax = axes[1]
ax.bar(range(len(decile_stats)), decile_stats['n'], color=PURPLE, edgecolor='none', alpha=0.75)
ax.set_xticks(range(len(decile_stats)))
ax.set_xticklabels(decile_stats['score_range'], rotation=45, ha='right', fontsize=8)
ax.set_title('Nº de tokens en cada decil de score')
ax.set_xlabel('Score ML (rango)')
ax.set_ylabel('Nº tokens')

plt.tight_layout()
plt.savefig('charts/17_winrate_buckets.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Threshold — encontrar el punto de operación para el sniper

In [ ]:
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, best_proba)

# Threshold objetivo: precision >= 3x base rate
target_prec = base_rate * 3
valid_mask  = prec_arr[:-1] >= target_prec
if valid_mask.any():
    # El threshold más bajo que mantiene precision >= target
    threshold_2x = thresh_arr[valid_mask][0]
    prec_at_th   = prec_arr[:-1][valid_mask][0]
    rec_at_th    = rec_arr[:-1][valid_mask][0]
else:
    threshold_2x = 0.5
    prec_at_th   = base_rate
    rec_at_th    = 1.0

# También calcular threshold F0.5 (precision-biased)
f05 = (1 + 0.5**2) * (prec_arr * rec_arr) / (0.5**2 * prec_arr + rec_arr + 1e-12)
f05_idx  = np.argmax(f05[:-1])
threshold_f05 = thresh_arr[f05_idx]

print(f'Base rate:                    {base_rate*100:.1f}%')
print(f'Objetivo 3x base (precision): {target_prec*100:.1f}%')
print()
print(f'Threshold 3x base:   {threshold_2x:.3f}')
print(f'  Precision:         {prec_at_th*100:.1f}%   (de cada 10 señales, ~{prec_at_th*10:.0f} son correctas)')
print(f'  Recall:            {rec_at_th*100:.1f}%   (detecta {rec_at_th*100:.0f}% de los pumps reales)')
print()
print(f'Threshold F0.5:      {threshold_f05:.3f}')
print(f'  Precision:         {prec_arr[:-1][f05_idx]*100:.1f}%')
print(f'  Recall:            {rec_arr[:-1][f05_idx]*100:.1f}%')

# Guardar el threshold para producción
FINAL_THRESHOLD = threshold_2x

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(thresh_arr, prec_arr[:-1], color=GREEN, lw=2, label='Precision')
ax.plot(thresh_arr, rec_arr[:-1],  color=RED,   lw=2, label='Recall')
ax.axhline(target_prec, color=YELLOW, ls=':', lw=1.2, label=f'Objetivo 3x base ({target_prec*100:.0f}%)')
ax.axvline(FINAL_THRESHOLD, color='white', ls='--', lw=1.5,
           label=f'Threshold = {FINAL_THRESHOLD:.3f}')
ax.set_title('Precision y Recall vs Threshold')
ax.set_xlabel('Threshold de decisión')
ax.set_ylabel('Score')
ax.legend()
ax.grid(alpha=0.2)

ax = axes[1]
# Nº de señales emitidas a cada threshold
signal_counts = [(best_proba >= t).sum() for t in thresh_arr]
ax2 = ax.twinx()
ax.plot(thresh_arr, prec_arr[:-1] * 100, color=GREEN, lw=2, label='Precision %')
ax2.plot(thresh_arr, signal_counts, color=PURPLE, lw=1.5, ls='--', label='Nº señales')
ax.axvline(FINAL_THRESHOLD, color='white', ls='--', lw=1.2)
ax.set_title('Precision vs Nº de señales emitidas')
ax.set_xlabel('Threshold')
ax.set_ylabel('Precision (%)', color=GREEN)
ax2.set_ylabel('Nº tokens señalados', color=PURPLE)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('charts/14_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Importancia de features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    idx = np.argsort(imp)[::-1]
    colors_bar = [GREEN if imp[i] > np.mean(imp) else ACCENT for i in idx]
    ax.barh([FEATURE_COLS[i] for i in idx], imp[idx], color=colors_bar, edgecolor='none')
    ax.set_title(f'Importancia de features — {best_name}')
    ax.set_xlabel('Importancia')
    ax.invert_yaxis()

ax = axes[1]
perm = permutation_importance(best_model, X_test, y_test, n_repeats=15,
                               random_state=42, n_jobs=-1, scoring='average_precision')
pidx = np.argsort(perm.importances_mean)[::-1]
colors_p = [GREEN if perm.importances_mean[i] > 0 else RED for i in pidx]
ax.barh([FEATURE_COLS[i] for i in pidx], perm.importances_mean[pidx],
        xerr=perm.importances_std[pidx], color=colors_p, edgecolor='none', capsize=3)
ax.axvline(0, color='white', lw=0.8)
ax.set_title(f'Permutation Importance (PR-AUC) — {best_name}\n'
             f'(verde = mejora PR-AUC, rojo = no aporta o daña)')
ax.set_xlabel('Δ PR-AUC al permutar la feature')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('charts/13_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Backtest — ¿qué hubiera pasado filtrando por el modelo?

In [ ]:
df_bt = df_s.iloc[split:].copy().reset_index(drop=True)
df_bt['ml_score']  = best_proba
df_bt['ml_signal'] = (best_proba >= FINAL_THRESHOLD).astype(int)
df_bt['ratio_1h']  = df_bt['outcome_price_1h'] / (df_bt['price_usd'] + eps)

all_t   = df_bt
signals = df_bt[df_bt['ml_signal'] == 1]

print('=' * 56)
print(f'  BACKTEST — threshold={FINAL_THRESHOLD:.3f}')
print('=' * 56)
print(f'  Tokens en test:         {len(all_t):,}')
print(f'  Señales emitidas:       {len(signals):,} ({len(signals)/len(all_t)*100:.1f}% de los tokens)')
print()
print(f'  Sin filtro ML (entrar en todo):')
print(f'    Win rate (+20%):      {(all_t["ratio_1h"]>=1.2).mean()*100:.1f}%')
print(f'    Ratio medio 1h:       {all_t["ratio_1h"].mean():.3f}x')
print(f'    Perder >50%:          {(all_t["ratio_1h"]<=0.5).mean()*100:.1f}%')
if len(signals) > 0:
    print()
    print(f'  Con filtro ML:')
    print(f'    Win rate (+20%):      {(signals["ratio_1h"]>=1.2).mean()*100:.1f}%  '
          f'(lift {(signals["ratio_1h"]>=1.2).mean()/max((all_t["ratio_1h"]>=1.2).mean(),0.001):.1f}x)')
    print(f'    Ratio medio 1h:       {signals["ratio_1h"].mean():.3f}x')
    print(f'    Perder >50%:          {(signals["ratio_1h"]<=0.5).mean()*100:.1f}%')
    print(f'    Mejor trade:          +{(signals["ratio_1h"].max()-1)*100:.0f}%')
print('=' * 56)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
bins = np.linspace(0, 4, 60)
ax.hist(all_t['ratio_1h'].clip(0,4), bins=bins, alpha=0.45, density=True,
        color=RED,   label=f'Sin filtro (n={len(all_t):,})')
if len(signals) > 0:
    ax.hist(signals['ratio_1h'].clip(0,4), bins=bins, alpha=0.75, density=True,
            color=GREEN, label=f'Filtrado ML (n={len(signals):,})')
ax.axvline(1.0, color='white', ls='--', lw=0.8, label='Sin cambio')
ax.axvline(1.2, color=YELLOW, ls='--', lw=1, label='+20% (TP)')
ax.set_title('Backtest — distribución de ratios 1h')
ax.set_xlabel('Ratio precio_1h / precio_entrada')
ax.set_ylabel('Densidad')
ax.legend()

ax = axes[1]
# Acumulado: win rate a medida que bajamos el threshold
thresholds_sweep = np.linspace(best_proba.max(), best_proba.min(), 100)
wr_list, n_list = [], []
for t in thresholds_sweep:
    mask = best_proba >= t
    n_list.append(mask.sum())
    wr_list.append((y_test[mask] == 1).mean() if mask.sum() > 0 else 0)

ax.plot(n_list, [w*100 for w in wr_list], color=ACCENT, lw=2)
ax.axhline(base_rate*100, color='white', ls='--', lw=0.8, label=f'Base rate {base_rate*100:.1f}%')
ax.axhline(target_prec*100, color=GREEN, ls=':', lw=1, label=f'3x base {target_prec*100:.1f}%')
if len(signals) > 0:
    ax.axvline(len(signals), color=YELLOW, ls='--', lw=1,
               label=f'Threshold actual ({len(signals)} señales)')
ax.set_title('Win rate vs nº de señales emitidas\n(cuantas menos señales → más precisas)')
ax.set_xlabel('Nº de tokens señalados')
ax.set_ylabel('Win rate (%)')
ax.legend(fontsize=8)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('charts/16_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Guardar modelo

In [ ]:
model_path = MODELS_DIR / f'pump_predictor_{best_name.lower()}.joblib'
joblib.dump(best_model, model_path)

meta = {
    'model_name':      best_name,
    'model_file':      model_path.name,
    'feature_cols':    FEATURE_COLS,
    'threshold':       float(FINAL_THRESHOLD),
    'label':           LABEL_NAME,
    'label_threshold': float(LABEL_THRESHOLD),
    'roc_auc':         float(results[best_name]['auc']),
    'pr_auc':          float(results[best_name]['ap']),
    'base_rate':       float(base_rate),
    'train_samples':   int(len(X_train)),
    'test_samples':    int(len(X_test)),
}

meta_path = MODELS_DIR / 'model_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('=' * 52)
print('  RESUMEN FINAL')
print('=' * 52)
for name, r in results.items():
    lift = r['ap'] / base_rate
    print(f'  {name:<20}  ROC-AUC={r["auc"]:.3f}  PR-AUC={r["ap"]:.3f}  lift={lift:.1f}x')
print(f'\n  Mejor:      {best_name}')
print(f'  Threshold:  {FINAL_THRESHOLD:.4f}')
print(f'  Base rate:  {base_rate*100:.1f}%  →  objetivo precision >= {target_prec*100:.0f}%')
print(f'  Guardado:   {model_path.name}')
print('=' * 52)
print()
print('Próximo paso: integrar en ai_analyzer.py')